# SFT do classificador de escopo LicitaGym — Tunix + LoRA (Gemma 3 1B)

Treina o Gemma 3 1B-it para ler a descrição de um item de licitação (e o objeto) e responder
`{"categoria": "forte|fraco|piso|borracha|obra_piso|manutencao|fora"}`.

**Antes de rodar**
1. Gere `train/val/test.jsonl` localmente com `training/export_dataset.py` (só rótulos revisados) e suba para
   `MyDrive/licitagym-treino/`. Nenhuma chave do Supabase entra aqui.
2. Aceite a licença do `google/gemma-3-1b-it` no Hugging Face e crie o Secret `HF_TOKEN` no Colab.
3. Runtime: **TPU v5e-1** (o v2-8 saiu do Colab grátis em set/2025) ou **GPU T4**.

API conferida no código do Tunix (commit `96431d6`, 30/09/2026) e no notebook oficial `examples/qlora_gemma.ipynb`.
Rode as células em ordem; reinicie o runtime depois da instalação.

In [ ]:
# 1) Instalação (mesmo caminho do notebook oficial). REINICIE o runtime ao terminar.
%pip install -q kagglehub safetensors tensorboard transformers grain datasets pyyaml
%pip install -q git+https://github.com/jax-ml/jax
%pip install -q git+https://github.com/google/tunix
%pip install -q git+https://github.com/google/qwix
%pip uninstall -q flax -y
%pip install -q git+https://github.com/google/flax
%pip install -q 'numpy>2'

In [ ]:
# 2) Código do repo (comum.py/avaliar.py), Drive com os dados e config
import os, sys, json, yaml
from google.colab import drive, userdata
drive.mount('/content/drive')
!git clone -q https://github.com/n4rch6pjt9-rgb/P-s-Treinamento-LLM /content/repo 2>/dev/null || git -C /content/repo pull -q
sys.path.insert(0, '/content/repo/training')
from comum import ROTULOS, ler_resposta
cfg = yaml.safe_load(open('/content/repo/training/config/sft_classificador.yaml'))
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
D = cfg['dados']['dir']
ler = lambda n: [json.loads(l) for l in open(f'{D}/{n}.jsonl', encoding='utf-8')]
train_records, val_records, test_records = ler('train'), ler('val'), ler('test')
print(len(train_records), len(val_records), len(test_records))

In [ ]:
# 3) Modelo base (safetensors do HF) como NNX + tokenizer
import jax, jax.numpy as jnp, numpy as np, optax, qwix
from flax import nnx
from huggingface_hub import snapshot_download
from orbax import checkpoint as ocp
from tunix.generate import sampler as sampler_lib
from tunix.generate import tokenizer_adapter as tokenizer_lib
from tunix.models.gemma3 import model as gemma3_model_lib
from tunix.models.gemma3 import params_safetensors as params_safetensors_lib
from tunix.models.gemma3 import params as gemma_params
from tunix.sft import metrics_logger, peft_trainer, utils

local_model_path = snapshot_download(repo_id=cfg['modelo']['hf_id'], ignore_patterns=['*.pth'])
EOS_TOKENS = json.load(open(os.path.join(local_model_path, 'generation_config.json'))).get('eos_token_id', [])
EOS_TOKENS = EOS_TOKENS if isinstance(EOS_TOKENS, list) else [EOS_TOKENS]
model_config = gemma3_model_lib.ModelConfig.gemma3_1b_it()

NUM_DEV = len(jax.devices())
print(jax.devices())
MESH = [(NUM_DEV, 1) if NUM_DEV > 1 else (1, 1), ('fsdp', 'tp')]  # 1 dispositivo: (1,1), como no notebook oficial
mesh = jax.make_mesh(*MESH, axis_types=(jax.sharding.AxisType.Auto,) * len(MESH[0]))
with jax.set_mesh(mesh):
    base_model = params_safetensors_lib.create_model_from_safe_tensors(local_model_path, model_config, mesh)

tokenizer = tokenizer_lib.Tokenizer(tokenizer_path=cfg['modelo']['tokenizer'])
if tokenizer.eos_id() not in EOS_TOKENS:
    EOS_TOKENS.append(tokenizer.eos_id())

In [ ]:
# 4) LoRA (qwix)
L = cfg['lora']
lora_provider = qwix.LoraProvider(module_path=L['module_path'], rank=L['rank'], alpha=float(L['alpha']))
lora_model = qwix.apply_lora_to_model(base_model, lora_provider, rngs=nnx.Rngs(params=0), **base_model.get_model_input())
with jax.set_mesh(mesh):
    state = nnx.state(lora_model)
    nnx.update(lora_model, jax.lax.with_sharding_constraint(state, nnx.get_partition_spec(state)))

In [ ]:
# 5) Dataset: loss só nos tokens da resposta (input_mask=1 na resposta)
import grain
TEMPLATE = {'prefix': '<start_of_turn>user\n', 'suffix': '\n<end_of_turn>\n<start_of_turn>model\n'}
MAX_LEN, BATCH = cfg['dados']['max_len'], cfg['dados']['batch']

class Tok(grain.MapTransform):
    def map(self, ex):
        src = tokenizer.tokenize(ex['prompt'], prefix=TEMPLATE['prefix'], suffix=TEMPLATE['suffix'], add_eos=False)
        dst = tokenizer.tokenize(ex['response'], add_eos=True)
        return src, dst

class Build(grain.MapTransform):
    def map(self, t):
        src, dst = t
        toks = np.concatenate([src, dst])[:MAX_LEN]
        mask = np.concatenate([np.zeros_like(src, bool), np.ones_like(dst, bool)])[:MAX_LEN]
        pad = MAX_LEN - len(toks)
        return peft_trainer.TrainingInput(
            input_tokens=np.pad(toks, (0, pad), constant_values=tokenizer.pad_id()),
            input_mask=np.pad(mask, (0, pad), constant_values=0))

def loader(records, epochs):
    return grain.DataLoader(
        data_source=records,
        sampler=grain.IndexSampler(num_records=len(records), num_epochs=epochs, shard_options=grain.NoSharding(), shuffle=True, seed=0),
        operations=[Tok(), Build(), grain.Batch(batch_size=BATCH, drop_remainder=True)])

train_ds, eval_ds = loader(train_records, cfg['dados']['epocas']), loader(val_records, 1)

def gen_model_input_fn(x: peft_trainer.TrainingInput):
    pad_mask = x.input_tokens != tokenizer.pad_id()
    return {'input_tokens': x.input_tokens, 'input_mask': x.input_mask,
            'positions': utils.build_positions_from_mask(pad_mask),
            'attention_mask': utils.make_causal_attn_mask(pad_mask)}

# prompts maiores que MAX_LEN perdem a resposta no corte: confira
cortados = sum(len(tokenizer.tokenize(r['prompt'])) + 12 > MAX_LEN for r in train_records)
print('exemplos que passam de MAX_LEN:', cortados)

In [ ]:
# 6) Treino (checkpoints LoRA no Drive; rodar de novo retoma do último step)
T = cfg['treino']
training_config = peft_trainer.TrainingConfig(
    eval_every_n_steps=T['eval_every_n_steps'], max_steps=T['max_steps'],
    metrics_logging_options=metrics_logger.MetricsLoggerOptions(log_dir='/tmp/tensorboard/lora', flush_every_n_steps=20),
    checkpoint_root_directory=T['checkpoint_dir'],
    checkpointing_options=ocp.CheckpointManagerOptions(save_interval_steps=T['save_interval_steps'], max_to_keep=T['max_to_keep']))
trainer = peft_trainer.PeftTrainer(lora_model, optax.adamw(float(T['lr'])), training_config).with_gen_model_input_fn(gen_model_input_fn)
with jax.set_mesh(mesh):
    trainer.train(train_ds, eval_ds)

In [ ]:
# 7) Avaliação no teste: predições -> preds_sft.jsonl e métricas (macro-F1)
A = cfg['avaliacao']
sampler = sampler_lib.Sampler(transformer=lora_model, tokenizer=tokenizer,
    cache_config=sampler_lib.CacheConfig(cache_size=MAX_LEN + A['max_generation_steps'] + 8, num_layers=model_config.num_layers,
                                         num_kv_heads=model_config.num_kv_heads, head_dim=model_config.head_dim))
preds = []
for i in range(0, len(test_records), BATCH):
    lote = test_records[i:i + BATCH]
    out = sampler(input_strings=[TEMPLATE['prefix'] + r['prompt'] + TEMPLATE['suffix'] for r in lote],
                  max_generation_steps=A['max_generation_steps'], max_prompt_length=MAX_LEN, eos_tokens=EOS_TOKENS, temperature=0.0)
    preds += [{'id': r['id'], 'saida': t} for r, t in zip(lote, out.text)]
with open(f'{D}/preds_sft.jsonl', 'w', encoding='utf-8') as f:
    for p in preds: f.write(json.dumps(p, ensure_ascii=False) + '\n')
!python /content/repo/training/avaliar.py --gabarito {D}/test.jsonl --pred sft={D}/preds_sft.jsonl

**Critério de aceite:** gere o baseline de regras localmente (precisa do repo LicitaGym) e compare:

```bash
python training/avaliar.py --gabarito data/test.jsonl --pred sft=data/preds_sft.jsonl \
  --baseline-regras ../LicitaGym/services/coletor-externo
```
Só exporte/use o modelo se `macro_f1(sft) > macro_f1(regras)`.

In [ ]:
# 8) Exporta o modelo com LoRA mesclado (safetensors) para o Drive
gemma_params.save_lora_merged_model_as_safetensors(
    local_model_path=local_model_path, output_dir=A['saida_merged'], lora_model=lora_model,
    rank=L['rank'], alpha=float(L['alpha']))
print('salvo em', A['saida_merged'])